# Gera Dados — Infográfico de Cotas

## Objetivo

Gerar a base de dados para o **infográfico de cotas** (racial e PCD), identificando os servidores admitidos por concurso público após 31/10/2023 e classificando-os pelo tipo de cota utilizada no ingresso.

## Fontes de dados

| Tabela | Uso |
| --- | --- |
| `VW001_TABELAO_SERV_202605.csv` (Volume bronze) | Dados cadastrais do servidor: órgão, cargo, raça/cor, situação, ocorrência de ingresso |
| `mgi-ouro.bd_siape.fatoservidor` | Tipo de cota (`co_tipo_cota`), CPF, datas de ingresso/exclusão |

## Filtros aplicados

- **Data de ingresso no órgão** > 31/10/2023 (marco legal das cotas)
- **Ocorrência de ingresso**: apenas concurso público/nomeação efetiva
- **Situação**: exclui servidores com `VAR_0001_SITUACAO = 'EXCLUIDO'`

## Classificação do tipo de cota

| `co_tipo_cota` | Classificação |
| --- | --- |
| 0 | Não Informado |
| 1 | Não (ampla concorrência) |
| 2 | Cota Racial |
| 3+ | Cota PCD |

## Lógica de join

```
fatoservidor (filtrado por data ingresso > 2023-10-31)
  INNER JOIN tabelao_csv (filtrado por concurso + não excluído)
    ON co_orgao_servidor = CO_ORGAO AND nu_matricula_servidor = MAT_SERV
```

## Saídas

| Destino | Formato | Consumido por |
| --- | --- | --- |
| `mgi-ouro.bd_coest.cotas_infografico` | Delta (overwrite) | Consultas ad-hoc / backup |
| `data/data.csv` | CSV | `index.Rmd` (agrupado por órgão) |
| `data/subdata.csv` | CSV | `index.Rmd` (agrupado por órgão + cargo) |

## Pipeline completo (Run All)

```
1. Conexão Spark + Parâmetros
2. Leitura Tabelão CSV (Volume bronze)
3. Leitura fatoservidor (camada ouro)
4. Join + Validação
5. Collect + Salvar Delta (camada ouro)
6. Agregação (lógica data_raw.R integrada) → data.csv + subdata.csv
7. Commit + git push (manual)
8. Renderizar index.Rmd (RStudio / Posit Connect)
```

## Estrutura do repositório git

```
Cotas_raciais/
├── Gera-dados-infografico-cotas   ← ESTE NOTEBOOK (gera dados + agregação)
├── data_raw.R                     ← Referência histórica (lógica incorporada ao notebook)
├── index.Rmd                      ← Infográfico HTML (reactable)
├── index.html                     ← Output renderizado
├── data/
│   ├── data.csv                          (agregado por órgão)
│   └── subdata.csv                       (agregado por órgão+cargo)
└── .gitignore
```

## Requisitos

- Cluster clássico com suporte a R (ex: `Cluster_MGI_prd_2026_language_R`)
- Serverless **não suporta** R

In [0]:
library(sparklyr)
library(dplyr)

sc <- spark_connect(method = "databricks")
cat("✅ Conexão Spark estabelecida\n")

In [0]:
# ==============================================================================
# PARÂMETROS
# ==============================================================================
# Competência do Tabelão CSV (ajustar conforme necessário)
tabelao <- '/Volumes/mgi-bronze/raw_data_volumes/mgi/DIGID/CGINF/01.Bases_SAS/COEST/tabelao_csv/VW001_TABELAO_SERV_202606.csv'

# Data-corte: marco legal das cotas (Lei 12.990/2014 renovada em 2023)
data_corte_ingresso <- '2023-10-31'

cat(paste0("Tabelão: ", tabelao, "\n"))
cat(paste0("Data corte ingresso: ", data_corte_ingresso, "\n"))

### 1. Tabelão CSV (dados cadastrais)

Leitura do arquivo `VW001_TABELAO_SERV` com as informações cadastrais do servidor.
Filtros aplicados:
- Ocorrência de ingresso no órgão: apenas concurso público/nomeação efetiva
- Situação diferente de EXCLUÍDO

In [0]:
# ==============================================================================
# OPÇÃO 1: Leitura via SQL read_files (recomendada — evita bug do sparklyr)
# ==============================================================================
df_tabelao <- tbl(sc, sql(paste0(
  "SELECT * FROM read_files('", tabelao, "', format => 'csv', header => true, delimiter => ';')"
))) |> 
  dplyr::filter(
    NO_OCORRENCIA_INGORG %in% c(
      'ADMISSAO POR CONCURSO PUBLICO',
      'ADMISSAO POR CONCURSO/EMPRESA',
      'NOMEACAO CARATER EFETIVO,ART.9,ITEM I ,LEI 8112/90'
    ),
    VAR_0001_SITUACAO != 'EXCLUIDO'
  )

# ==============================================================================
# OPÇÃO 2: Leitura via spark_read_csv (API nativa sparklyr)
# Nota: pode falhar com bug interno em certos DBRs (spark_csv_is_embedded).
#       Descomentar abaixo e comentar a Opção 1 caso queira testar.
# ==============================================================================
# df_tabelao <- spark_read_csv(
#   sc,
#   name      = "tabelao_serv",
#   path      = tabelao,
#   header    = TRUE,
#   delimiter = ";",
#   memory    = FALSE
# ) |> 
#   dplyr::filter(
#     NO_OCORRENCIA_INGORG %in% c(
#       'ADMISSAO POR CONCURSO PUBLICO',
#       'ADMISSAO POR CONCURSO/EMPRESA',
#       'NOMEACAO CARATER EFETIVO,ART.9,ITEM I ,LEI 8112/90'
#     ),
#     VAR_0001_SITUACAO != 'EXCLUIDO'
#   )


In [0]:
# Selecionar apenas as colunas necessárias para o join e o infográfico
df_tab <- df_tabelao |> dplyr::select(
    COMPET,
    VAR_0001_SITUACAO,
    REGIME_JUR_E_SIT, 
    NO_SIT_SERV, 
    CO_GRUPO_E_CARGO_ORIGEM, 
    NO_CARGO_ORIGEM, 
    NO_COR_ORIGEM_ETNICA,   # raça/cor (para cruzar com cota racial)
    CO_ORGAO,               # chave de join
    MAT_SERV,               # chave de join
    NO_ORGAO, 
    NO_OCORRENCIA_INGORG
)

### 2. fatoservidor (tipo de cota)

Leitura da tabela `mgi-ouro.bd_siape.fatoservidor` — contém o campo `co_tipo_cota` que identifica se o servidor ingressou por cota racial, PCD ou ampla concorrência.

Filtro: `da_ocor_ingr_orgao_serv > '2023-10-31'` (marco legal)

In [0]:
# Leitura da fatoservidor: servidores com ingresso no órgão após o marco legal
df <- tbl(sc, sql(paste0(
  "SELECT * FROM `mgi-ouro`.`bd_siape`.`fatoservidor` ",
  "WHERE da_ocor_ingr_orgao_serv > '", data_corte_ingresso, "'"
)))

cat(paste0("Registros na fatoservidor (pós ", data_corte_ingresso, "): "))
df |> count()

In [0]:
# Selecionar campos e criar classificação do tipo de cota
df_fato <- df |> 
  select(
    co_orgao_servidor,          # chave de join
    nu_matricula_servidor,      # chave de join
    nu_cpf, 
    da_ocor_ingr_orgao_serv, 
    da_ocor_exclusao_serv, 
    co_tipo_cota  
  ) |> 
  mutate(
    gr_mat = paste0(co_orgao_servidor, nu_matricula_servidor),
    tipo_cota = case_when(
      co_tipo_cota == 0 ~ 'Não Informado',
      co_tipo_cota == 1 ~ 'Não',
      co_tipo_cota == 2 ~ 'Cota Racial',
      TRUE ~ 'Cota PCD'
    )
  )

### 3. Join e gravação

Cruzamento das duas fontes (fatoservidor × Tabelão) por `co_orgao + matricula`.

O resultado é gravado como tabela Delta em **`mgi-ouro.bd_coest.cotas_infografico`** (mode = overwrite).

In [0]:
# Join entre fatoservidor e Tabelão por órgão + matrícula
# INNER JOIN garante apenas servidores presentes em ambas as fontes
df_join <- df_fato |> 
  inner_join(df_tab, by = c("co_orgao_servidor" = "CO_ORGAO", 
                            "nu_matricula_servidor" = "MAT_SERV"))

In [0]:
# Validação: total de registros no resultado final
cat("Total de registros no df_join: ")
df_join |> count()

In [0]:
# Exibe tabela interativa com opção de download (seta ↓ no canto do resultado)
display(df_join)

In [0]:
# ==============================================================================
# Collect do Spark para R local (df_join tem ~8k linhas, cabe na memória)
# ==============================================================================
df_local <- df_join |> collect()
cat(paste0("✅ Collect concluído: ", nrow(df_local), " linhas\n"))

# ==============================================================================
# Salvar como tabela Delta na camada ouro
# ==============================================================================
sdf_register(df_join, "temp_cotas_infografico")
DBI::dbExecute(sc, "CREATE OR REPLACE TABLE `mgi-ouro`.`bd_coest`.`cotas_infografico` AS SELECT * FROM temp_cotas_infografico")
cat("✅ Tabela Delta salva em mgi-ouro.bd_coest.cotas_infografico\n")


In [0]:
# ==============================================================================
# AGREGAÇÃO dos dados para o infográfico (lógica originalmente em data_raw.R)
# Usa df_local (já em memória vindo do collect() da célula anterior)
# ==============================================================================
library(readr)
library(dplyr)

# Adicionar flag de cota racial (1/0)
cotas <- df_local %>% mutate(cota = ifelse(tipo_cota == "Cota Racial", 1, 0))

# Sumarizar por Órgão + Cargo
subdata <- cotas %>%
  group_by(NO_ORGAO, NO_CARGO_ORIGEM) %>%
  summarise(count_gr_mat = n_distinct(gr_mat),
            cotas = sum(cota),
            Percentual.cotas = round(cotas / count_gr_mat, 2),
            qte_20 = count_gr_mat * 0.2,
            diferenca = qte_20 - cotas,
            .groups = "drop")

# Sumarizar por Órgão
data <- cotas %>%
  group_by(NO_ORGAO) %>%
  summarise(count_gr_mat = n_distinct(gr_mat),
            cotas = sum(cota),
            Percentual.cotas = round(cotas / count_gr_mat, 2),
            qte_20 = count_gr_mat * 0.2,
            diferenca = qte_20 - cotas,
            .groups = "drop") |>
  ungroup() |>
  mutate(NO_CARGO_ORIGEM = '') |>
  select(NO_ORGAO, NO_CARGO_ORIGEM, everything())

# Gravar CSVs no diretório data/ do repositório git
write_csv(subdata, "data/subdata.csv")
write_csv(data, "data/data.csv")

cat("✅ Agregação concluída (lógica data_raw.R integrada)\n")
cat(paste0("   subdata.csv: ", nrow(subdata), " linhas\n"))
cat(paste0("   data.csv: ", nrow(data), " linhas\n"))
cat("   Local: data/ (repositório git)\n")

In [0]:
# ==============================================================================
# RENDERIZAR index.Rmd (executar localmente — requer pandoc)
# Descomentar abaixo para rodar no RStudio/máquina local
# ==============================================================================
# library(rmarkdown)
# render("index.Rmd", output_file = "index.html", quiet = TRUE)
# cat(paste0("✅ index.html renderizado! Tamanho: ", round(file.size("index.html") / 1024, 1), " KB\n"))